# Score stage-2 `C3 > C2` pairs by alignment potential

This notebook scores the stage-2 format-alignment pairs so you can select the pairs that are most useful to train on.

The ICML 2025 Alignment Potential idea is:

```text
score = explicit reward margin - current model implicit margin
```

For our categorical data, `C3 > C2` has the same explicit class margin for every pair, so the ranking is mainly driven by the current model's implicit margin:

```text
implicit margin = normalized_logp(chosen) - normalized_logp(rejected)
```

Pairs with small absolute implicit margin are the pairs the current model has not already separated. Those are usually more useful than pairs the model already strongly prefers.


In [ ]:
from pathlib import Path
import os, sys, json, math, re, gc
from difflib import SequenceMatcher

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
PROJECT_ROOT = Path("/home/dinh-tuan/Documents/Project/BioASQ_MRES/Task-Structured Counterfactual Preference Mining")
sys.path.insert(0, str(PROJECT_ROOT))

import cse_dpo.train_factoid_three_stage_dpo as three_stage
from transformers import AutoTokenizer

# Point this at the run whose stage-1 best adapter should define the current model margin.
# Change to the 3B run when scoring the 3B model's stage-2 data.
RUN_ROOT = PROJECT_ROOT / "Artifacts/cse_dpo/three_stage_dpo_qwen25_05b_c3_c1_curriculum_semantic_full_bp4096"
# RUN_ROOT = PROJECT_ROOT / "Artifacts/cse_dpo/three_stage_dpo_qwen25_3b_c3_c1_curriculum_semantic_full_bp2048"

STAGE2_JSONL = PROJECT_ROOT / "Artifacts/cse_dpo/class_judgments/candidate_banks_train_first400_c3_c1_v1_gold_c3/staged_curriculum_pairs/dpo_stage2_format_alignment_all_pairs.jsonl"
MODEL_PRESET = "qwen25_3b" if "qwen25_3b" in str(RUN_ROOT) else "qwen25_05b"

# If True, load the model and compute current-model log-prob margins.
# If False, the notebook still writes a metadata-only audit CSV, but no real alignment-potential ranking.
COMPUTE_MODEL_LOGPROBS = True

MAX_LENGTH = 4096
ALLOW_TRUNCATION = False
APPEND_EOS = True
BATCH_SIZE = 1
SELECT_RATIO = 0.50  # top 50% by alignment-potential score

OUTPUT_ROOT = PROJECT_ROOT / f"Artifacts/cse_dpo/alignment_potential_scores/stage2_{MODEL_PRESET}_{RUN_ROOT.name}"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# The stage-2 reference/current model should be the best stage-1 adapter.
stage1_manifest_path = RUN_ROOT / "stage_1_concept_learning" / "manifest.json"
if not stage1_manifest_path.exists():
    raise FileNotFoundError(stage1_manifest_path)
stage1_manifest = json.loads(stage1_manifest_path.read_text())
ADAPTER_FOR_MARGIN = Path(stage1_manifest.get("best_selected_adapter", stage1_manifest.get("best_mrr_adapter", stage1_manifest.get("adapter"))))
if not ADAPTER_FOR_MARGIN.exists():
    raise FileNotFoundError(ADAPTER_FOR_MARGIN)

print(json.dumps({
    "model_preset": MODEL_PRESET,
    "run_root": str(RUN_ROOT),
    "stage2_jsonl": str(STAGE2_JSONL),
    "adapter_for_margin": str(ADAPTER_FOR_MARGIN),
    "compute_model_logprobs": COMPUTE_MODEL_LOGPROBS,
    "max_length": MAX_LENGTH,
    "select_ratio": SELECT_RATIO,
    "output_root": str(OUTPUT_ROOT),
}, indent=2))


In [ ]:
def read_jsonl(path: Path):
    rows = []
    with path.open(encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return rows

def write_jsonl(path: Path, rows):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

def norm_text(s):
    s = str(s or "").lower()
    s = re.sub(r"\[/?b?e\]", " ", s)
    s = re.sub(r"[^a-z0-9]+", " ", s)
    return " ".join(s.split())

def text_similarity(a, b):
    return SequenceMatcher(None, norm_text(a), norm_text(b)).ratio()

CLASS_RANK = {"C1": 0, "C2": 1, "C3": 2}

def zscore(values):
    arr = np.asarray(values, dtype=float)
    mu = np.nanmean(arr)
    sigma = np.nanstd(arr)
    if not np.isfinite(sigma) or sigma == 0:
        return np.zeros_like(arr, dtype=float)
    return (arr - mu) / sigma

rows = read_jsonl(STAGE2_JSONL)
print(f"Loaded {len(rows):,} stage-2 pairs from {STAGE2_JSONL}")

base_records = []
for r in rows:
    chosen_rank = CLASS_RANK.get(r.get("chosen_class"), np.nan)
    rejected_rank = CLASS_RANK.get(r.get("rejected_class"), np.nan)
    chosen_text = r.get("chosen_candidate") or r.get("chosen")
    rejected_text = r.get("rejected_candidate") or r.get("rejected")
    base_records.append({
        "pair_id": r.get("pair_id"),
        "question_id": r.get("question_id"),
        "question": r.get("question"),
        "gold_aliases": " | ".join(map(str, r.get("gold_aliases", []))) if isinstance(r.get("gold_aliases"), list) else r.get("gold_aliases"),
        "chosen_candidate": chosen_text,
        "rejected_candidate": rejected_text,
        "chosen": r.get("chosen"),
        "rejected": r.get("rejected"),
        "chosen_class": r.get("chosen_class"),
        "rejected_class": r.get("rejected_class"),
        "chosen_raw_class": r.get("chosen_raw_class"),
        "rejected_raw_class": r.get("rejected_raw_class"),
        "explicit_class_margin": chosen_rank - rejected_rank,
        "string_similarity": text_similarity(chosen_text, rejected_text),
        "chosen_len_chars": len(str(chosen_text or "")),
        "rejected_len_chars": len(str(rejected_text or "")),
        "chosen_evidence_support": r.get("chosen_evidence_support"),
        "rejected_evidence_support": r.get("rejected_evidence_support"),
        "chosen_source_model": r.get("chosen_source_model"),
        "rejected_source_model": r.get("rejected_source_model"),
        "chosen_response_id": r.get("chosen_response_id"),
        "rejected_response_id": r.get("rejected_response_id"),
        "stage_rule": r.get("stage_rule"),
    })

df = pd.DataFrame(base_records)
df.head()


In [ ]:
# Optional: compute current-model implicit margins.
# This is the expensive part. It scores chosen and rejected completions under the stage-1 best adapter.
if COMPUTE_MODEL_LOGPROBS:
    cfg = three_stage.Config(
        model_preset=MODEL_PRESET,
        max_length=MAX_LENGTH,
        batch_size=BATCH_SIZE,
        append_eos_to_completions=APPEND_EOS,
        allow_truncated_examples=ALLOW_TRUNCATION,
        drop_truncated_examples=not ALLOW_TRUNCATION,
        lora_dropout=None,
    )
    tokenizer = AutoTokenizer.from_pretrained(str(ADAPTER_FOR_MARGIN), trust_remote_code=True, use_fast=True)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    encoded = []
    dropped = []
    for r in tqdm(rows, desc="Tokenizing stage-2 pairs"):
        try:
            item = three_stage._encode_pair(
                tokenizer,
                r,
                MAX_LENGTH,
                append_eos=APPEND_EOS,
                allow_truncation=ALLOW_TRUNCATION,
            )
            encoded.append(item)
        except Exception as exc:
            dropped.append({"pair_id": r.get("pair_id"), "question_id": r.get("question_id"), "error": str(exc)})

    print(f"Encoded {len(encoded):,}; dropped {len(dropped):,}")
    if dropped:
        pd.DataFrame(dropped).to_csv(OUTPUT_ROOT / "dropped_stage2_pairs_for_margin.csv", index=False)
        write_jsonl(OUTPUT_ROOT / "dropped_stage2_pairs_for_margin.jsonl", dropped)

    model = three_stage._load_policy(cfg, ADAPTER_FOR_MARGIN)
    scores = three_stage._reference_scores(model, encoded, tokenizer, cfg)

    scored = {}
    for item, (chosen_logp, rejected_logp) in zip(encoded, scores):
        pair_id = item["row"]["pair_id"]
        chosen_tokens = int(item["chosen_tokens"])
        rejected_tokens = int(item["rejected_tokens"])
        chosen_logp_norm = float(chosen_logp) / max(1, chosen_tokens)
        rejected_logp_norm = float(rejected_logp) / max(1, rejected_tokens)
        scored[pair_id] = {
            "chosen_logp_sum": float(chosen_logp),
            "rejected_logp_sum": float(rejected_logp),
            "chosen_tokens": chosen_tokens,
            "rejected_tokens": rejected_tokens,
            "prompt_tokens": int(item["prompt_tokens"]),
            "chosen_logp_norm": chosen_logp_norm,
            "rejected_logp_norm": rejected_logp_norm,
            "implicit_margin_sum": float(chosen_logp) - float(rejected_logp),
            "implicit_margin_norm": chosen_logp_norm - rejected_logp_norm,
            "abs_implicit_margin_norm": abs(chosen_logp_norm - rejected_logp_norm),
            "model_prefers_chosen": chosen_logp_norm > rejected_logp_norm,
        }

    for key, default in [
        ("chosen_logp_sum", np.nan), ("rejected_logp_sum", np.nan),
        ("chosen_tokens", np.nan), ("rejected_tokens", np.nan), ("prompt_tokens", np.nan),
        ("chosen_logp_norm", np.nan), ("rejected_logp_norm", np.nan),
        ("implicit_margin_sum", np.nan), ("implicit_margin_norm", np.nan),
        ("abs_implicit_margin_norm", np.nan), ("model_prefers_chosen", np.nan),
    ]:
        df[key] = df["pair_id"].map(lambda x: scored.get(x, {}).get(key, default))

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
else:
    print("Skipping model log-prob scoring. Set COMPUTE_MODEL_LOGPROBS=True for real alignment-potential scoring.")


In [ ]:
# Compute alignment-potential style scores.
# If model logprobs were computed, this follows the released-code intuition:
#   z(explicit margin) - z(abs implicit margin)
# Since stage 2 is almost entirely C3>C2, explicit margin is usually constant;
# useful ranking mostly comes from lower abs_implicit_margin_norm.
if "abs_implicit_margin_norm" in df.columns and df["abs_implicit_margin_norm"].notna().any():
    df["explicit_margin_z"] = zscore(df["explicit_class_margin"].fillna(df["explicit_class_margin"].median()))
    df["abs_implicit_margin_z"] = zscore(df["abs_implicit_margin_norm"].fillna(df["abs_implicit_margin_norm"].median()))
    df["alignment_potential_score"] = df["explicit_margin_z"] - df["abs_implicit_margin_z"]
    df["signed_gap_score"] = df["explicit_class_margin"] - df["implicit_margin_norm"].fillna(0.0)
else:
    # Metadata fallback: useful for audit only, not a faithful MAP score.
    df["explicit_margin_z"] = zscore(df["explicit_class_margin"].fillna(df["explicit_class_margin"].median()))
    df["abs_implicit_margin_z"] = np.nan
    df["alignment_potential_score"] = df["explicit_margin_z"]
    df["signed_gap_score"] = np.nan

# Additional audit flags. High similarity C3>C2 examples are often risky because C2 may be a clean synonym.
df["near_duplicate_text"] = df["string_similarity"] >= 0.85
df["rejected_longer_than_chosen"] = df["rejected_len_chars"] > df["chosen_len_chars"]
df["chosen_supported_or_gold"] = df["chosen_evidence_support"].isin(["supported", "not_judged"])
df["rejected_supported"] = df["rejected_evidence_support"].eq("supported")

ranked = df.sort_values(
    ["alignment_potential_score", "string_similarity"],
    ascending=[False, True],
    na_position="last",
).reset_index(drop=True)
ranked["alignment_potential_rank"] = np.arange(1, len(ranked) + 1)
ranked["selected_top_ratio"] = ranked["alignment_potential_rank"] <= max(1, int(round(SELECT_RATIO * len(ranked))))

csv_path = OUTPUT_ROOT / "stage2_alignment_potential_scored.csv"
jsonl_path = OUTPUT_ROOT / "stage2_alignment_potential_scored.jsonl"
ranked.to_csv(csv_path, index=False)
write_jsonl(jsonl_path, ranked.replace({np.nan: None}).to_dict(orient="records"))

selected_ids = set(ranked.loc[ranked["selected_top_ratio"], "pair_id"])
selected_rows = [r for r in rows if r.get("pair_id") in selected_ids]
selected_jsonl = OUTPUT_ROOT / f"dpo_stage2_format_alignment_top{int(SELECT_RATIO*100)}_alignment_potential.jsonl"
write_jsonl(selected_jsonl, selected_rows)

summary = {
    "source_stage2_jsonl": str(STAGE2_JSONL),
    "run_root": str(RUN_ROOT),
    "adapter_for_margin": str(ADAPTER_FOR_MARGIN),
    "model_preset": MODEL_PRESET,
    "compute_model_logprobs": COMPUTE_MODEL_LOGPROBS,
    "pairs_total": len(rows),
    "pairs_scored_with_logprobs": int(ranked["implicit_margin_norm"].notna().sum()) if "implicit_margin_norm" in ranked else 0,
    "select_ratio": SELECT_RATIO,
    "selected_pairs": len(selected_rows),
    "csv_path": str(csv_path),
    "jsonl_path": str(jsonl_path),
    "selected_jsonl": str(selected_jsonl),
}
(OUTPUT_ROOT / "summary.json").write_text(json.dumps(summary, indent=2) + "\n", encoding="utf-8")
print(json.dumps(summary, indent=2))
ranked.head(20)


## How to use the scored file

The main audit file is `stage2_alignment_potential_scored.csv`.

Useful columns:

- `implicit_margin_norm`: current model's normalized preference for chosen over rejected.
- `abs_implicit_margin_norm`: how strongly the current model separates the pair already.
- `alignment_potential_score`: higher means more useful under the paper's idea.
- `model_prefers_chosen`: `False` means the current model still prefers the C2 answer over C3.
- `near_duplicate_text`: `True` means the pair may be risky because C2 is textually very close to C3.

The selected JSONL file can be used as a smaller stage-2 training file. It keeps the top `SELECT_RATIO` by `alignment_potential_score`.
